# Motif Visualization Demo

This notebook demonstrates cool visual examples for consensus sequence searching and structural motifs. We'll create:

## Structural Motif Visualizations  
- **3D interactive views** - Using py3Dmol to visualize catalytic triads and other motifs
- **Biotite structure analysis** - Using biotite for structural analysis and visualization
- **Distance geometry** - Showing constraint distances between motif atoms

---
## Setup and Imports

In [ ]:
import sys
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import warnings
warnings.filterwarnings('ignore')

# Add parent directories to path
sys.path.insert(0, os.path.abspath('../sequence_motif'))
sys.path.insert(0, os.path.abspath('../structure_motif'))

# Import our tools
from motif_searcher import expand_motif, AMINO_ACID_NOMENCLATURE
from search_3d_motif import parse_motif_file, search_single_file

# Optional: py3Dmol for 3D visualization
try:
    import py3Dmol
    HAS_PY3DMOL = True
    print(" py3Dmol available for 3D visualization")
except ImportError:
    HAS_PY3DMOL = False
    print(" py3Dmol not available (install with: pip install py3Dmol)")

# Optional: biotite for structure analysis
try:
    import biotite.structure as struc
    import biotite.structure.io.pdb as pdb
    import biotite.structure.graphics as graphics
    HAS_BIOTITE = True
    print(" biotite available for structure analysis")
except ImportError:
    HAS_BIOTITE = False
    print(" biotite not available (install with: pip install biotite)")

# Directory paths
PROTEIN_FILES_DIR = '../protein_files'
MOTIFS_DIR = '../structure_motif/motifs'

print("\nSetup complete!")

---
# Part 1: Structural Motif Visualizations

Now let's visualize 3D structural motifs using py3Dmol and biotite.

## 1.1 Catalytic Triad Search and Visualization

In [ ]:
# First, let's search for catalytic triads in our protein files
catalytic_triad_def = parse_motif_file(os.path.join(MOTIFS_DIR, 'catalytic_triad.json'))

print("Catalytic Triad Motif Definition:")
print("="*50)
print(f"Name: {catalytic_triad_def['motif_name']}")
print(f"Description: {catalytic_triad_def['description']}")
print(f"\nComponents:")
for comp in catalytic_triad_def['components']:
    print(f"   {comp['id']}: {comp['residue_type']}")
    for name, atom in comp['atom_selectors'].items():
        print(f"      {name} -> {atom}")
print(f"\nConstraints:")
for c in catalytic_triad_def['constraints']:
    print(f"   {c['type']}: {c['atoms'][0]} <-> {c['atoms'][1]} = {c['value']}±{c['tolerance']}Å")

In [ ]:
# Search for catalytic triads in a serine protease (1AQ7 is trypsin)
pdb_file = os.path.join(PROTEIN_FILES_DIR, '1AQ7.pdb')

if os.path.exists(pdb_file):
    found_motifs = search_single_file(pdb_file, catalytic_triad_def)
    
    print(f"\nSearching for catalytic triad in 1AQ7 (Trypsin)...")
    print(f"Found {len(found_motifs)} catalytic triad(s)!")
    
    for i, motif in enumerate(found_motifs):
        print(f"\n Triad {i+1}:")
        for res in motif['residues']:
            print(f"      {res['res_name']}-{res['chain_id']}-{res['res_id']}")
else:
    print(f"PDB file not found: {pdb_file}")
    found_motifs = []

## 1.2 Interactive 3D Visualization with py3Dmol

In [ ]:
def visualize_motif_3d(pdb_path, motif_residues, view_style='cartoon'):
    """
    Create an interactive 3D visualization of a protein with highlighted motif residues.
    
    Args:
        pdb_path: Path to PDB file
        motif_residues: List of dicts with 'chain_id' and 'res_id'
        view_style: 'cartoon', 'stick', or 'surface'
    """
    if not HAS_PY3DMOL:
        print(" py3Dmol not available. Install with: pip install py3Dmol")
        return None
    
    with open(pdb_path, 'r') as f:
        pdb_data = f.read()
    
    view = py3Dmol.view(width=800, height=600)
    view.addModel(pdb_data, 'pdb')
    
    # Style the whole protein as transparent cartoon
    view.setStyle({view_style: {'color': 'lightgrey', 'opacity': 0.7}})
    
    # Highlight motif residues with different colors
    colors = ['red', 'blue', 'green', 'orange', 'purple', 'cyan']
    
    for i, res in enumerate(motif_residues):
        color = colors[i % len(colors)]
        
        # Highlight as sticks
        selection = {'chain': res['chain_id'], 'resi': res['res_id']}
        view.addStyle(selection, {'stick': {'color': color, 'radius': 0.3}})
        
        # Add label
        label_text = f"{res.get('res_name', '?')}{res['res_id']}"
        view.addLabel(label_text, {
            'backgroundColor': color,
            'fontColor': 'white',
            'fontSize': 14,
            'position': {'x': 0, 'y': 0, 'z': 0}
        }, selection)
    
    view.zoomTo()
    view.spin('y', speed=0.5)
    
    return view

# Visualize the catalytic triad
if found_motifs and HAS_PY3DMOL:
    print("Interactive 3D View of Catalytic Triad in Trypsin (1AQ7)")
    print("(The view will spin - hover to interact)")
    view = visualize_motif_3d(pdb_file, found_motifs[0]['residues'])
    if view:
        view.show()
elif not HAS_PY3DMOL:
    print("Install py3Dmol for 3D visualization: pip install py3Dmol")

## 1.3 Structural Analysis with Biotite

In [ ]:
def analyze_motif_geometry(pdb_path, motif_residues):
    """
    Analyze the geometry of a structural motif using biotite.
    """
    if not HAS_BIOTITE:
        print(" biotite not available. Install with: pip install biotite")
        return None
    
    # Read structure
    pdb_file_obj = pdb.PDBFile.read(pdb_path)
    structure = pdb.get_structure(pdb_file_obj, model=1)
    atoms = structure[structure.hetero == False]  # Exclude heteroatoms
    
    print("\n" + "="*60)
    print("STRUCTURAL MOTIF GEOMETRY ANALYSIS")
    print("="*60)
    
    # Get coordinates for motif residues
    motif_coords = {}
    for res in motif_residues:
        mask = (atoms.res_id == res['res_id']) & (atoms.chain_id == res['chain_id'])
        res_atoms = atoms[mask]
        
        res_key = f"{res.get('res_name', 'UNK')}{res['res_id']}"
        motif_coords[res_key] = {}
        
        print(f"\n {res_key}:")
        for atom_name in ['N', 'CA', 'C', 'O', 'CB', 'OG', 'ND1', 'OD1', 'OD2', 'SG', 'NZ']:
            atom_mask = res_atoms.atom_name == atom_name
            if np.any(atom_mask):
                coord = res_atoms.coord[atom_mask][0]
                motif_coords[res_key][atom_name] = coord
                print(f"      {atom_name}: ({coord[0]:.2f}, {coord[1]:.2f}, {coord[2]:.2f})")
    
    # Calculate distances between key atoms
    print("\n" + "-"*40)
    print("INTER-RESIDUE DISTANCES:")
    print("-"*40)
    
    res_keys = list(motif_coords.keys())
    for i, key1 in enumerate(res_keys):
        for key2 in res_keys[i+1:]:
            # Get CA atoms
            if 'CA' in motif_coords[key1] and 'CA' in motif_coords[key2]:
                dist = np.linalg.norm(motif_coords[key1]['CA'] - motif_coords[key2]['CA'])
                print(f"   {key1} CA <-> {key2} CA: {dist:.2f} Å")
    
    return motif_coords

# Analyze geometry
if found_motifs and HAS_BIOTITE:
    coords = analyze_motif_geometry(pdb_file, found_motifs[0]['residues'])

## 1.4 Distance Diagram Visualization

In [ ]:
def plot_motif_distances(motif_residues, constraints):
    """
    Create a schematic diagram showing the distance constraints in a motif.
    """
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.set_xlim(-2, 12)
    ax.set_ylim(-2, 10)
    ax.axis('off')
    
    # Position residues in a triangle/circle
    n_res = len(motif_residues)
    angles = np.linspace(0, 2*np.pi, n_res, endpoint=False) - np.pi/2
    radius = 3
    center = (5, 5)
    
    positions = {}
    colors = ['#E74C3C', '#3498DB', '#2ECC71', '#F39C12', '#9B59B6']
    
    for i, res in enumerate(motif_residues):
        x = center[0] + radius * np.cos(angles[i])
        y = center[1] + radius * np.sin(angles[i])
        res_key = res.get('res_name', 'UNK')
        positions[res_key] = (x, y)
        
        # Draw residue circle
        circle = plt.Circle((x, y), 0.8, color=colors[i % len(colors)], alpha=0.8, zorder=10)
        ax.add_patch(circle)
        
        # Add residue label
        ax.text(x, y, res_key, ha='center', va='center', fontsize=12, 
               fontweight='bold', color='white', zorder=11)
        ax.text(x, y - 1.3, f"Res {res['res_id']}", ha='center', va='center', 
               fontsize=9, color='#666')
    
    # Draw distance constraints
    for constraint in constraints:
        if constraint['type'] == 'distance':
            # Parse constraint atoms
            atoms = constraint['atoms']
            
            # Get residue types from component ids (simplified mapping)
            comp_to_res = {'ser': 'SER', 'his': 'HIS', 'asp': 'ASP', 'cys1': 'CYS', 'cys2': 'CYS', 'lys': 'LYS'}
            
            res1 = comp_to_res.get(atoms[0].split('.')[0], 'UNK')
            res2 = comp_to_res.get(atoms[1].split('.')[0], 'UNK')
            
            if res1 in positions and res2 in positions:
                x1, y1 = positions[res1]
                x2, y2 = positions[res2]
                
                # Draw line
                ax.plot([x1, x2], [y1, y2], 'k-', linewidth=2, alpha=0.6, zorder=5)
                
                # Add distance label
                mid_x, mid_y = (x1 + x2) / 2, (y1 + y2) / 2
                dist_text = f"{constraint['value']}±{constraint['tolerance']}Å"
                ax.text(mid_x + 0.5, mid_y + 0.5, dist_text, fontsize=10, 
                       bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    ax.set_title('Structural Motif Distance Constraints\n(Catalytic Triad)', 
                fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

# Create distance diagram
if found_motifs:
    plot_motif_distances(found_motifs[0]['residues'], catalytic_triad_def['constraints'])

In [ ]:
def compare_motifs_across_structures(pdb_files, motif_def, max_structures=5):
    """
    Search for a motif across multiple structures and compare results.
    """
    results = []
    
    print(f"Searching for '{motif_def['motif_name']}' across {min(len(pdb_files), max_structures)} structures...\n")
    
    for pdb_path in pdb_files[:max_structures]:
        if os.path.exists(pdb_path):
            fname = os.path.basename(pdb_path)
            found = search_single_file(pdb_path, motif_def)
            results.append({
                'file': fname,
                'found': len(found),
                'motifs': found
            })
            status = f" {len(found)} found" if found else " None"
            print(f"   {fname}: {status}")
    
    # Create visualization
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Bar chart of motif counts
    files = [r['file'].split('.')[0] for r in results]
    counts = [r['found'] for r in results]
    colors = ['#27AE60' if c > 0 else '#BDC3C7' for c in counts]
    
    bars = ax1.bar(files, counts, color=colors, edgecolor='white', linewidth=2)
    ax1.set_xlabel('Structure', fontsize=12)
    ax1.set_ylabel('Motifs Found', fontsize=12)
    ax1.set_title(f"Motif Detection Results\n{motif_def['motif_name']}", fontsize=14, fontweight='bold')
    ax1.tick_params(axis='x', rotation=45)
    
    # Add count labels on bars
    for bar, count in zip(bars, counts):
        if count > 0:
            ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
                    str(count), ha='center', va='bottom', fontweight='bold')
    
    # Pie chart of hit rate
    hits = sum(1 for c in counts if c > 0)
    misses = len(counts) - hits
    
    ax2.pie([hits, misses], labels=['Hits', 'No Match'], autopct='%1.0f%%',
           colors=['#27AE60', '#BDC3C7'], startangle=90, explode=(0.05, 0))
    ax2.set_title('Detection Rate', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    return results

# Search across all available structures
pdb_files = sorted([os.path.join(PROTEIN_FILES_DIR, f) for f in os.listdir(PROTEIN_FILES_DIR) 
                   if f.endswith(('.pdb', '.cif'))])

if pdb_files:
    comparison_results = compare_motifs_across_structures(pdb_files, catalytic_triad_def, max_structures=10)